# BinSense — M1: Download Dataset from S3

Downloads the **3,875-bin subset** of the Amazon Bin Image Dataset.

| Asset | S3 path | Local destination |
|---|---|---|
| Metadata JSON | `s3://aft-vbi-pds/metadata/{id}.json` | `data/metadata/` |
| Bin image JPEG | `s3://aft-vbi-pds/bin-images/{id}.jpg` | `data/images/` |

**No AWS credentials needed** — bucket is public/anonymous.  
**Resume-safe** — re-running this notebook skips files already on disk.

In [1]:
# Cell 1: Bootstrap — path resolution + Colab setup
#
# Code (utils/, tools/, notebooks/) lives in the git repo.
# Data (images/, metadata/, splits/) lives on Google Drive.
# BINSENSE_DATA_DIR tells env_utils where the data root is when they differ.

import sys, os
from pathlib import Path

# ── Colab branch ──────────────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')

    _DRIVE_ROOT = '/content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense'

    # Code root: once the GitHub remote is set, replace with a git clone:
    #   !git clone https://github.com/<user>/binsense /content/binsense
    #   PROJECT_ROOT = Path('/content/binsense')
    PROJECT_ROOT = Path(_DRIVE_ROOT)

    # Data root — always on Drive (images + metadata are not in git)
    os.environ['BINSENSE_DATA_DIR'] = str(Path(_DRIVE_ROOT) / 'data')

    import subprocess
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + ['boto3', 'tqdm'],
        check=True,
    )

# ── Local branch (D:/GitHub_Repo/Amazon BinSense) ────────────────────────
except ImportError:
    IN_COLAB = False
    if os.getenv('BINSENSE_DIR'):
        PROJECT_ROOT = Path(os.environ['BINSENSE_DIR'])
    else:
        _cwd = Path.cwd()
        PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    # Set BINSENSE_DATA_DIR if data lives elsewhere, e.g.:
    #   os.environ['BINSENSE_DATA_DIR'] = r'G:\My Drive\...\data'
    # Leave unset to fall back to PROJECT_ROOT/data.

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Running in  : {"Google Colab" if IN_COLAB else "Local"}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA DIR    : {os.getenv("BINSENSE_DATA_DIR", str(PROJECT_ROOT / "data"))}')
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in  : Google Colab
PROJECT_ROOT: /content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense


In [2]:
# ── Cell 2: Project imports + path config ────────────────────────────────
#
# cfg.base_dir  is the single source of truth for all data paths.
# CSV_PATH and downloader destinations all derive from cfg — never from
# PROJECT_ROOT — so local and Colab stay in sync automatically.

from utils.env_utils import setup_env
from tools.download import BinS3Downloader

cfg = setup_env(verbose=True)

# Derive all paths from cfg.base_dir (env_utils is the one source of truth)
CSV_PATH = cfg.base_dir / 'analysis' / 'unique_image_names.csv'

print(f'\nCSV          : {CSV_PATH}')
print(f'Metadata dir : {cfg.metadata_dir}')
print(f'Images dir   : {cfg.images_dir}')

assert CSV_PATH.exists(), (
    f"❌ CSV not found at {CSV_PATH}\n"
    f"   Make sure unique_image_names.csv is in the analysis/ folder."
)

[BinSense] Running in: Google Colab
[BinSense] Base dir  : /content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense

CSV          : /content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense/analysis/unique_image_names.csv
Metadata dir : /content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense/data/metadata
Images dir   : /content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense/data/images


## Step 1 — Load bin IDs

In [3]:
# ── Cell 3: Load and inspect the ID list ─────────────────────────────────
ids = BinS3Downloader.load_ids(CSV_PATH)

print(f'Total IDs loaded : {len(ids)}')
print(f'First 5          : {ids[:5]}')
print(f'Last  5          : {ids[-5:]}')
print(f'Short (5-digit)  : {[i for i in ids if len(i) == 5][:5]}')
print(f'Long  (6-digit)  : {[i for i in ids if len(i) == 6][:5]}')
print(f'Sample meta key  : metadata/{ids[0]}.json')
print(f'Sample image key : bin-images/{ids[0]}.jpg')

Total IDs loaded : 3875
First 5          : ['113526', '113528', '113529', '113549', '113550']
Last  5          : ['00964', '00965', '00971', '00972', '00973']
Short (5-digit)  : ['11363', '11364', '11365', '11369', '11370']
Long  (6-digit)  : ['113526', '113528', '113529', '113549', '113550']
Sample meta key  : metadata/113526.json
Sample image key : bin-images/113526.jpg


## Step 2 — Download metadata JSONs

~3,875 small JSON files (~1–5 KB each).  
Expected time: **1–3 min** at 32 parallel threads.

In [11]:
# ── Cell 4: Download metadata ─────────────────────────────────────────────
dl = BinS3Downloader(
    meta_dir=cfg.metadata_dir,
    images_dir=cfg.images_dir,
    workers=10, # Only 10 workers are allowed by S3 before we start getting throttled
)

meta_stats = dl.download_metadata(ids)
BinS3Downloader.print_stats(meta_stats, label='Metadata')

Metadata JSONs:   0%|          | 0/3875 [00:00<?, ?file/s]

[Metadata]  total=3875  downloaded=0  skipped(resume)=3875  failed=0


## Step 3 — Download bin images

3,875 JPEG images (~200–500 KB each ≈ ~1 GB total).  
Expected time: **5–15 min** depending on connection speed.  
`workers=10` keeps memory pressure low on Colab T4.

In [10]:
# ── Cell 5: Download images ───────────────────────────────────────────────
image_stats = dl.download_images(ids, workers=10)
BinS3Downloader.print_stats(image_stats, label='Images')

Bin images  :   0%|          | 0/3875 [00:00<?, ?file/s]

[Images]  total=3875  downloaded=0  skipped(resume)=3875  failed=0


## Step 4 — Verify & auto-retry any failures

In [8]:
# ── Cell 6: Verify completeness + auto-retry ──────────────────────────────
v = dl.verify(ids)
BinS3Downloader.print_verify(v)

# Auto-retry anything still missing (idempotent — won't re-download present files)
if v['meta_missing'] > 0:
    print(f'\nRe-trying {v["meta_missing"]} missing metadata files...')
    BinS3Downloader.print_stats(
        dl.download_metadata(v['meta_missing_ids']),
        label='Metadata retry'
    )

if v['images_missing'] > 0:
    print(f'\nRe-trying {v["images_missing"]} missing image files...')
    BinS3Downloader.print_stats(
        dl.download_images(v['images_missing_ids'], workers=10),
        label='Images retry'
    )

Verification  (3875 IDs checked)
  Metadata :  3875 present  |      0 missing
  Images   :  3875 present  |      0 missing


In [9]:
# ── Cell 7: Final disk summary ────────────────────────────────────────────
meta_files  = sorted(cfg.metadata_dir.glob('*.json'))
image_files = sorted(cfg.images_dir.glob('*.jpg'))

meta_bytes  = sum(f.stat().st_size for f in meta_files)
image_bytes = sum(f.stat().st_size for f in image_files)

print('=== Download summary ===')
print(f'  Metadata : {len(meta_files):>5} files   {meta_bytes/1e6:>8.1f} MB')
print(f'  Images   : {len(image_files):>5} files   {image_bytes/1e6:>8.1f} MB')
print(f'  Total    :         {(meta_bytes + image_bytes)/1e9:.2f} GB on disk')
print(f'  Saved to : {cfg.base_dir / "data"}')

if len(meta_files) == len(ids) and len(image_files) == len(ids):
    print('\n✅ All 3,875 bins present — ready for M2 (EDA + splits).')
else:
    n_missing = len(ids) - min(len(meta_files), len(image_files))
    print(f'\n⚠️  {n_missing} bins still incomplete — re-run Cell 6 to retry.')

=== Download summary ===
  Metadata :  3874 files        7.9 MB
  Images   :  3874 files      267.5 MB
  Total    :         0.28 GB on disk
  Saved to : /content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense/data

⚠️  1 bins still incomplete — re-run Cell 6 to retry.


In [ ]:
!pip install label-studio